# SENTINEL-GNSS — Colab Training Notebook

**Runtime:** GPU → Runtime ▸ Change runtime type ▸ T4 GPU

**Workflow:**
1. Mount Google Drive (data lives there)
2. Clone / pull latest code from GitHub
3. Install dependencies
4. Run feature prep → train → evaluate

**One-time setup:** Upload `data/labelled/sentinel_gnss_labelled.csv` to  
`My Drive/sentinel-gnss/data/labelled/` in Google Drive.

## 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify the labelled CSV is present
import os
DRIVE_DATA = '/content/drive/MyDrive/sentinel-gnss'
assert os.path.exists(f'{DRIVE_DATA}/data/labelled/sentinel_gnss_labelled.csv'), \
    'Upload sentinel_gnss_labelled.csv to My Drive/sentinel-gnss/data/labelled/ first!'
print('Drive mounted and data file found.')

## 2 — Clone / update code from GitHub

In [ ]:
import os

GITHUB_REPO = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # ← change this
REPO_DIR    = '/content/sentinel-gnss'

if os.path.exists(REPO_DIR):
    # Already cloned — just pull latest changes
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {GITHUB_REPO} {REPO_DIR}
    %cd {REPO_DIR}

print(f'Working directory: {os.getcwd()}')

## 3 — Install dependencies

In [ ]:
# PyTorch with CUDA 12.1 is already installed on Colab T4.
# Only install the extra packages not pre-installed.
!pip install -q imbalanced-learn
print('Dependencies installed.')

## 4 — Link Drive data into the repo directory structure

In [ ]:
import os, shutil

REPO_DIR   = '/content/sentinel-gnss'
DRIVE_DATA = '/content/drive/MyDrive/sentinel-gnss'

# Create local dirs
os.makedirs(f'{REPO_DIR}/data/labelled',           exist_ok=True)
os.makedirs(f'{REPO_DIR}/data/processed/windows',  exist_ok=True)
os.makedirs(f'{REPO_DIR}/results/models/checkpoints', exist_ok=True)
os.makedirs(f'{REPO_DIR}/results/figures',         exist_ok=True)

# Symlink Drive data → repo (avoids copying large files)
src_csv = f'{DRIVE_DATA}/data/labelled/sentinel_gnss_labelled.csv'
dst_csv = f'{REPO_DIR}/data/labelled/sentinel_gnss_labelled.csv'
if not os.path.exists(dst_csv):
    os.symlink(src_csv, dst_csv)

# If windows are already cached on Drive, symlink them too (skips feature_prep)
for split in ('train', 'val', 'test'):
    src_npz = f'{DRIVE_DATA}/data/processed/windows/{split}.npz'
    dst_npz = f'{REPO_DIR}/data/processed/windows/{split}.npz'
    if os.path.exists(src_npz) and not os.path.exists(dst_npz):
        os.symlink(src_npz, dst_npz)
        print(f'  Linked {split}.npz from Drive cache.')

# Symlink checkpoint dir to Drive so checkpoints survive session end
drive_ckpt = f'{DRIVE_DATA}/results/models/checkpoints'
os.makedirs(drive_ckpt, exist_ok=True)
local_ckpt = f'{REPO_DIR}/results/models/checkpoints'
if os.path.isdir(local_ckpt) and not os.path.islink(local_ckpt):
    shutil.rmtree(local_ckpt)
if not os.path.islink(local_ckpt):
    os.symlink(drive_ckpt, local_ckpt)

# Same for figures
drive_figs = f'{DRIVE_DATA}/results/figures'
os.makedirs(drive_figs, exist_ok=True)
local_figs = f'{REPO_DIR}/results/figures'
if os.path.isdir(local_figs) and not os.path.islink(local_figs):
    shutil.rmtree(local_figs)
if not os.path.islink(local_figs):
    os.symlink(drive_figs, local_figs)

print('Data links ready.')

## 5 — Verify GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 6 — Build feature windows (one-time; skips if cached)

In [ ]:
%cd /content/sentinel-gnss
!python -m src.models.feature_prep

# Cache windows to Drive so this step is skipped on future sessions
import shutil, os
DRIVE_WINDOWS = '/content/drive/MyDrive/sentinel-gnss/data/processed/windows'
os.makedirs(DRIVE_WINDOWS, exist_ok=True)
for split in ('train', 'val', 'test'):
    src = f'/content/sentinel-gnss/data/processed/windows/{split}.npz'
    dst = f'{DRIVE_WINDOWS}/{split}.npz'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'  Cached {split}.npz to Drive.')

## 7 — Train

In [ ]:
%cd /content/sentinel-gnss
# Use --resume to continue from the last checkpoint if Colab disconnected
# Checkpoints are saved to Google Drive automatically (see symlink above)
!python -m src.models.train --resume --batch_size 256

## 8 — Evaluate

In [ ]:
%cd /content/sentinel-gnss
!python -m src.models.evaluate
# Figures are saved to Google Drive automatically (see symlink above)

## 9 — Show figures inline
Browse all saved figures without leaving Colab.

In [ ]:
import glob
from IPython.display import Image, display

figs = sorted(glob.glob('/content/sentinel-gnss/results/figures/*.png'))
print(f'{len(figs)} figures found:')
for f in figs:
    print(f'  {f}')
    display(Image(filename=f, width=900))